# 🔧 Feature Engineering — CMAPSS FD001

## Contexto

Os dois notebooks anteriores estabeleceram uma compreensão profunda do dataset: a análise exploratória mapeou o comportamento dos sensores e identificou quais variáveis carregam informação de degradação; a análise de sobrevivência caracterizou probabilisticamente a distribuição do tempo até a falha.

Este notebook traduz esses achados em decisões concretas de engenharia. Dados brutos raramente chegam prontos para um modelo de machine learning — sensores em escalas incompatíveis, targets mal definidos e sinais ruidosos degradam diretamente a qualidade das predições. Feature engineering é o processo de transformar os dados brutos em representações que o modelo consiga aprender de forma eficiente.

Todas as transformações aplicadas aqui são motivadas por evidências dos notebooks anteriores — não são escolhas arbitrárias, mas decisões fundamentadas na física do problema e no comportamento estatístico dos dados.

## Decisões que este notebook justifica e implementa

**1. Definição do target RUL com cap**
O RUL linear assume que o motor no ciclo 1 já está em degradação ativa — o que não é verdade. A survival analysis mostrou que nenhum motor falha antes do ciclo 128, e a EDA confirmou que os sensores permanecem estáveis na primeira metade da vida útil. Aplicamos um cap de 125 ciclos: acima desse limiar, o motor é considerado saudável e o RUL é tratado como constante.

**2. Normalização por motor (Min-Max individual)**
Normalizar pelo dataset inteiro vaza informação do futuro para o passado — o modelo veria, durante o treinamento, valores que só existem nos ciclos finais de outros motores. A normalização é feita individualmente por motor, preservando a integridade temporal dos dados.

**3. Rolling features (média e desvio padrão móveis)**
A análise de instabilidade do sinal (Seção 11 da EDA) demonstrou que o ruído dos sensores aumenta conforme a falha se aproxima. O desvio padrão móvel captura essa heterocedasticidade como feature explícita — informação que a média móvel sozinha não consegue representar.

**4. Exportação do dataset final**
O dataset processado é salvo em formato pronto para alimentar diretamente os modelos do próximo notebook — sem necessidade de reprocessamento.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.family'] = 'sans-serif'

In [2]:
COL_NAMES = [
    'unit_id', 'time_cycles',
    'op_setting_1', 'op_setting_2', 'op_setting_3',
    'sensor_temp_fan_inlet', 'sensor_temp_lpc_outlet', 'sensor_temp_hpc_outlet',
    'sensor_temp_lpt_outlet', 'sensor_pressure_inlet', 'sensor_pressure_fan_inlet',
    'sensor_pressure_ratio', 'sensor_physical_fan_speed', 'sensor_physical_core_speed',
    'sensor_engine_pressure_ratio', 'sensor_static_hpc_outlet', 'sensor_fuel_flow_ps30',
    'sensor_corrected_fan_speed', 'sensor_corrected_core_speed', 'sensor_bypass_ratio',
    'sensor_bleed_enthalpy', 'sensor_demanded_fan_speed', 'sensor_demanded_corrected_fan_speed',
    'sensor_hpt_coolant_bleed', 'sensor_lpt_coolant_bleed', 'sensor_bpt_ratio'
]

DROP_COLS = [
    'sensor_temp_fan_inlet', 'sensor_pressure_inlet', 'sensor_pressure_fan_inlet',
    'sensor_physical_fan_speed', 'sensor_engine_pressure_ratio', 'sensor_corrected_fan_speed',
    'sensor_bleed_enthalpy', 'sensor_demanded_corrected_fan_speed', 'sensor_hpt_coolant_bleed',
    'op_setting_1', 'op_setting_2', 'op_setting_3',
]

df_train = pd.read_csv('data/train_FD001.txt', sep='\s+', header=None, names=COL_NAMES)
df_test  = pd.read_csv('data/test_FD001.txt',  sep='\s+', header=None, names=COL_NAMES)
df_rul   = pd.read_csv('data/RUL_FD001.txt',   header=None, names=['rul'])

df_train = df_train.drop(columns=DROP_COLS)
df_test  = df_test.drop(columns=DROP_COLS)

SENSORS = [col for col in df_train.columns if col.startswith('sensor')]

print(f'Train shape: {df_train.shape}')
print(f'Test shape:  {df_test.shape}')
print(f'Sensores:    {len(SENSORS)}')

Train shape: (20631, 14)
Test shape:  (13096, 14)
Sensores:    12


In [3]:
RUL_CAP = 125  # limiar baseado na literatura CMAPSS e na survival analysis

# RUL linear
rul_max = df_train.groupby('unit_id')['time_cycles'].max().reset_index()
rul_max.columns = ['unit_id', 'max_cycles']
df_train = df_train.merge(rul_max, on='unit_id')
df_train['RUL'] = df_train['max_cycles'] - df_train['time_cycles']
df_train = df_train.drop(columns='max_cycles')

# Aplicando o cap — acima de 125 ciclos o motor é considerado saudável
df_train['RUL'] = df_train['RUL'].clip(upper=RUL_CAP)

print(f'RUL máximo após cap: {df_train["RUL"].max()}')
print(f'RUL mínimo:          {df_train["RUL"].min()}')
print(f'\nDistribuição do RUL com cap:')
print(df_train['RUL'].describe().round(2))

RUL máximo após cap: 125
RUL mínimo:          0

Distribuição do RUL com cap:
count    20631.00
mean        86.83
std         41.67
min          0.00
25%         51.00
50%        103.00
75%        125.00
max        125.00
Name: RUL, dtype: float64


In [4]:
# Normalização Min-Max por motor individualmente
# IMPORTANTE: normalizar pelo dataset inteiro vazaria informação do futuro —
# o modelo veria, durante treino, valores que só existem nos ciclos finais de outros motores.
# Normalizando por motor, cada série temporal é independente.

def minmax_by_unit(df, sensors):
    df = df.copy()
    for sensor in sensors:
        min_vals = df.groupby('unit_id')[sensor].transform('min')
        max_vals = df.groupby('unit_id')[sensor].transform('max')
        denom = max_vals - min_vals
        # Evitar divisão por zero em sensores constantes por motor
        df[sensor] = np.where(denom == 0, 0, (df[sensor] - min_vals) / denom)
    return df

df_train = minmax_by_unit(df_train, SENSORS)

print('Após normalização por motor:')
print(df_train[SENSORS].describe().round(3))

Após normalização por motor:
       sensor_temp_lpc_outlet  sensor_temp_hpc_outlet  sensor_temp_lpt_outlet  \
count               20631.000               20631.000               20631.000   
mean                    0.421                   0.435                   0.382   
std                     0.194                   0.194                   0.203   
min                     0.000                   0.000                   0.000   
25%                     0.284                   0.298                   0.234   
50%                     0.397                   0.417                   0.345   
75%                     0.540                   0.552                   0.497   
max                     1.000                   1.000                   1.000   

       sensor_pressure_ratio  sensor_physical_core_speed  \
count              20631.000                   20631.000   
mean                   0.607                       0.393   
std                    0.201                       0.242   
m

In [5]:
WINDOW = 30  # janela padrão da literatura CMAPSS

def add_rolling_features(df, sensors, window=WINDOW):
    df = df.copy()
    for sensor in sensors:
        # Média móvel — captura a tendência de degradação suavizando o ruído
        df[f'{sensor}_mean_{window}'] = (
            df.groupby('unit_id')[sensor]
            .transform(lambda x: x.rolling(window, min_periods=1).mean())
        )
        # Desvio padrão móvel — captura o aumento de instabilidade do sinal
        # conforme o motor se aproxima da falha (heterocedasticidade)
        df[f'{sensor}_std_{window}'] = (
            df.groupby('unit_id')[sensor]
            .transform(lambda x: x.rolling(window, min_periods=1).std().fillna(0))
        )
    return df

df_train = add_rolling_features(df_train, SENSORS)

ROLLING_FEATURES = [c for c in df_train.columns 
                    if '_mean_' in c or '_std_' in c]

print(f'Features originais:  {len(SENSORS)}')
print(f'Rolling features:    {len(ROLLING_FEATURES)}')
print(f'Total de features:   {len(SENSORS) + len(ROLLING_FEATURES)}')
print(f'\nShape do df_train:   {df_train.shape}')

Features originais:  12
Rolling features:    24
Total de features:   36

Shape do df_train:   (20631, 39)


In [6]:
# Aplicando as mesmas transformações no conjunto de teste
df_test = minmax_by_unit(df_test, SENSORS)
df_test = add_rolling_features(df_test, SENSORS)

# RUL do test vem do arquivo RUL_FD001.txt
# Cada valor corresponde ao RUL do último ciclo de cada motor no test
rul_test = df_rul['rul'].values
rul_test = np.clip(rul_test, 0, RUL_CAP)

print(f'Shape df_test:  {df_test.shape}')
print(f'Shape rul_test: {rul_test.shape}')
print(f'\nRUL test — distribuição:')
print(pd.Series(rul_test).describe().round(2))

Shape df_test:  (13096, 38)
Shape rul_test: (100,)

RUL test — distribuição:
count    100.00
mean      74.45
std       40.28
min        7.00
25%       32.75
50%       86.00
75%      112.25
max      125.00
dtype: float64


In [7]:
ALL_FEATURES = SENSORS + ROLLING_FEATURES

def create_sliding_windows(df, rul_series, window=WINDOW, is_test=False):
    X, y = [], []
    
    for unit_id, group in df.groupby('unit_id'):
        group = group.sort_values('time_cycles')
        data = group[ALL_FEATURES].values
        
        if is_test:
            # No test pegamos apenas a última janela de cada motor
            # — é o ponto onde precisamos predizer o RUL
            if len(data) >= window:
                X.append(data[-window:])
            else:
                # Motor com menos ciclos que a janela — padding com zeros à esquerda
                pad = np.zeros((window - len(data), len(ALL_FEATURES)))
                X.append(np.vstack([pad, data]))
        else:
            # No train geramos todas as janelas possíveis
            rul = group['RUL'].values
            for i in range(len(data) - window + 1):
                X.append(data[i:i+window])
                y.append(rul[i + window - 1])
    
    if is_test:
        return np.array(X)
    return np.array(X), np.array(y)

X_train_seq, y_train_seq = create_sliding_windows(df_train, None, is_test=False)
X_test_seq = create_sliding_windows(df_test, None, window=WINDOW, is_test=True)
y_test_seq = rul_test

print(f'X_train_seq: {X_train_seq.shape}  → (amostras, ciclos, features)')
print(f'y_train_seq: {y_train_seq.shape}')
print(f'X_test_seq:  {X_test_seq.shape}')
print(f'y_test_seq:  {y_test_seq.shape}')

X_train_seq: (17731, 30, 36)  → (amostras, ciclos, features)
y_train_seq: (17731,)
X_test_seq:  (100, 30, 36)
y_test_seq:  (100,)


In [8]:
import os

os.makedirs('data/processed', exist_ok=True)

# Dataset tabular (para XGBoost / Random Forest)
ALL_FEATURES = SENSORS + ROLLING_FEATURES

X_train_tab = df_train[ALL_FEATURES].values
y_train_tab = df_train['RUL'].values

# Para o test tabular pegamos o último ciclo de cada motor
X_test_tab = (df_test.groupby('unit_id')
              .apply(lambda x: x.sort_values('time_cycles').iloc[-1])
              [ALL_FEATURES].values)
y_test_tab = rul_test

# Salvando tudo
np.save('data/processed/X_train_tab.npy', X_train_tab)
np.save('data/processed/y_train_tab.npy', y_train_tab)
np.save('data/processed/X_test_tab.npy',  X_test_tab)
np.save('data/processed/y_test_tab.npy',  y_test_tab)

np.save('data/processed/X_train_seq.npy', X_train_seq)
np.save('data/processed/y_train_seq.npy', y_train_seq)
np.save('data/processed/X_test_seq.npy',  X_test_seq)
np.save('data/processed/y_test_seq.npy',  y_test_seq)

print('Datasets salvos em data/processed/')
print(f'\nTabular:')
print(f'  X_train: {X_train_tab.shape} | y_train: {y_train_tab.shape}')
print(f'  X_test:  {X_test_tab.shape}  | y_test:  {y_test_tab.shape}')
print(f'\nSequencial:')
print(f'  X_train: {X_train_seq.shape} | y_train: {y_train_seq.shape}')
print(f'  X_test:  {X_test_seq.shape}  | y_test:  {y_test_seq.shape}')

C:\Users\marce\AppData\Local\Temp\ipykernel_22620\2797113636.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sort_values('time_cycles').iloc[-1])


Datasets salvos em data/processed/

Tabular:
  X_train: (20631, 36) | y_train: (20631,)
  X_test:  (100, 36)  | y_test:  (100,)

Sequencial:
  X_train: (17731, 30, 36) | y_train: (17731,)
  X_test:  (100, 30, 36)  | y_test:  (100,)
